# 特征很多时怎样压缩并观察结构？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

训练矩阵 (133, 13) 测试矩阵 (45, 13)
{'n_features': 13, 'retained_components': 5, 'training_variance_retained': 0.802957184592555, 'test_reconstruction_mse': 0.24244722632633586, 'mean_baseline_mse': 1.1407206931576077}


## 1. 补全：逐特征重构误差
平均各特征误差，核对它与整体测试重构 MSE 相同。

In [2]:
assert np.isclose(comparison['standardized_mse'].mean(),reconstruction_mse)
print(comparison.sort_values('standardized_mse',ascending=False).head())

                feature  standardized_mse
8       proanthocyanins          0.502430
7  nonflavanoid_phenols          0.352682
0               alcohol          0.307955
1            malic_acid          0.258955
9       color_intensity          0.247876


## 2. 修改：只保留 2 维
训练集重新拟合 2 维 PCA，检查测试误差不能优于保留更多同一 PCA 方向的结果。

In [3]:
two = PCA(n_components=2).fit(train)
two_mse = float(np.mean((test-two.inverse_transform(two.transform(test)))**2))
assert two_mse >= reconstruction_mse - 1e-9
print('2 维测试重构 MSE',two_mse)

2 维测试重构 MSE 0.5686335202498786


## 3. 迁移：新观测恢复到原单位
对一个新观测执行标准化→降维→重构→还原单位，验证形状。

In [4]:
new = dataset.data.iloc[[test_idx[0]]]
recovered = scaler.inverse_transform(pca.inverse_transform(pca.transform(scaler.transform(new))))
assert recovered.shape == (1,13) and np.isfinite(recovered).all()
print(pd.DataFrame({'原值':new.iloc[0],'重构值':recovered[0]}))

                                  原值         重构值
alcohol                        13.48   13.284851
malic_acid                      1.81    2.029731
ash                             2.41    2.447550
alcalinity_of_ash              20.50   18.923188
magnesium                     100.00  102.808976
total_phenols                   2.70    2.868287
flavanoids                      2.98    2.896822
nonflavanoid_phenols            0.26    0.299349
proanthocyanins                 1.86    2.010350
color_intensity                 5.10    4.811677
hue                             1.04    1.068727
od280/od315_of_diluted_wines    3.47    3.187078
proline                       920.00  902.069754
